# 1. Joint $\zeta(3),\zeta(4)$ construction: matrices, sums and minors

**Paper:** CMF Theorem 1 (Theorem 6.1 (`thm:cmf1`)), the dual-pairing lemma (Lemma 6.6 (`prop:z34-connection`)), the limit lemma (Lemma 6.7 (`lem:z34-limits`)), the denominator and numerator presentations (Lemma 6.9 (`prop:z34-q-iv`), Lemma 6.10 (`prop:z34-p-iv`)), integer normalization (Lemma 6.11 (`prop:z34-int`)), the dual sum lemma (Lemma 6.12 (`lem: dual sum`)), and the second presentation (Lemma 6.13 (`prop:z34-dual`)).

This notebook reconstructs both trajectories with `ramanujantools`, verifies the pairing as an identity in $\mathbb Q(n)$, and explains the normalizations and determinant readouts. The second notebook supplies the recurrence certificates that turn the sum identities into statements for every $n$. The third supplies the analytic rate and nonvanishing arguments.


Run every cell from top to bottom in a fresh Python kernel. This notebook uses only the local Python modules and matrix data included beside it; no other notebook needs to run first. Install the bundle's `requirements.txt` once. Saved output is a record of a completed run, while every certificate is recomputed when you run the cells.


In [1]:
from pathlib import Path
import sys
# Jupyter normally starts in the notebook folder; the bundle root also works.
_candidates = [Path.cwd(), Path.cwd() / "joint"]
NOTEBOOK_DIR = next((p.resolve() for p in _candidates
                     if (p / "certificates.py").is_file() and (p / "data" / "dual_companion.json").is_file()), None)
if NOTEBOOK_DIR is None:
    raise RuntimeError("Start this notebook from its joint/ folder or the bundle root.")
sys.path.insert(0, str(NOTEBOOK_DIR))
if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
print("Local joint-proof inputs found.")


Local joint-proof inputs found.


The paper uses **row states** and right multiplication:
\[
\Phi_M(n)=M(0)\cdots M(n-1),\qquad \Phi_M(0)=I,
\qquad Y_{n+1}=Y_nM(n).
\]
We retain its notation throughout:

| Paper | Python | Meaning |
|---|---|---|
| $M(n)$ | `M` | $4\times4$ forward trajectory matrix |
| $Q(n)$ | `Q` | closed $3\times3$ dual trajectory block |
| $Q^\vee(n)$ | inverse transpose of `Q` | $Q(n)^{-\mathsf T}$ |
| $K(n),c(n)$ | `K`, `c` | dual pairing matrix and scalar |
| $q_n,p_{3,n},p_{4,n}$ | `R @ Phi_M` read through $e_0$ | original rational readouts |
| $C_n$ | `normalization(n)` | product of four binomial coefficients |
| $\mathbf q_n,\mathbf p_{3,n},\mathbf p_{4,n}$ | `forward_values(n)` | $(C_n/12)(q_n,p_{3,n},p_{4,n})$ |
| $B_{n,k},B'_{n,k}$ | `B(n,k)`, `Bprime(n,k)` | summand and its Gamma-interpolation derivative |
| $\mathbf a_n,\mathbf b_n,\mathbf d_n$ | rows of `dual_rows(n)` | coefficients of $\zeta(4),\zeta(3),1$ |
| $\Lambda_n$ | `Lam(n)` | factorial multiplier in the determinant presentation |

Bold and unbold readouts are different normalizations; the ratios $p_{s,n}/q_n=\mathbf p_{s,n}/\mathbf q_n$ are equal. The matrix companion $C(n)$ is unrelated to the scalar $C_n$.


## A. Trajectory matrices and the dual pairing

**Paper:** Lemma 6.6 (`prop:z34-connection`); the companion convention is Definition `def: companion form`. All matrix residuals below are symbolic identities in $n$, rather than checks at finitely many indices.

The supplied JSON matrices are reference inputs. We rebuild them from the stated CMF starting points and directions and compare every entry.


In [2]:
import sympy as sp
from ramanujantools import Matrix, Position
from ramanujantools.cmf import pFq, MeijerG
from common import n, matrix, data, expression, sm, equal, determinant, regular_nonnegative
M = matrix("matrices_for_zeta3_4.json", "M")
U = matrix("matrices_for_zeta3_4.json", "U")
C = matrix("matrices_for_zeta3_4.json", "C")
G = matrix("dual_matrices_zeta3_4.json", "G")
Q = Matrix(G[:3,:3])
print("M, U, C, Q shapes:", M.shape, U.shape, C.shape, Q.shape)

M, U, C, Q shapes: (4, 4) (4, 3) (3, 3) (3, 3)


### A1. Rebuild the two trajectories
These are exactly the starting points and directions in the proof section. RT's parameter axes are `x0,...,y0,...` for ${}_pF_q$ and `a0,...,b0,...` for Meijer $G$.

In [3]:
axes_F = sp.symbols("x0:5") + sp.symbols("y0:4")
s0 = (0,0,1,1,1, 1,1,1,1)
t0 = (-14,-15,13,14,12, 1,2,3,4)
MF = pFq(5,4,1).trajectory_matrix(Position(dict(zip(axes_F,t0))), Position(dict(zip(axes_F,s0))), n)
assert equal(MF,M)
print("PASS: forward RT trajectory equals supplied M(n).")

axes_G = sp.symbols("a0:5") + sp.symbols("b0:5")
s1 = (0,0,1,1,2, 0,0,0,0,0)
t1 = (-16,-17,11,12,10, -2,-1,0,1,2)
MG = MeijerG(5,2,5,5,1).trajectory_matrix(Position(dict(zip(axes_G,t1))), Position(dict(zip(axes_G,s1))), n)
assert equal(MG,G)
assert G[3,:] == Matrix([[0,0,0,-1]])
print("PASS: dual RT trajectory and closed 3 by 3 block Q(n).")

PASS: forward RT trajectory equals supplied M(n).


PASS: dual RT trajectory and closed 3 by 3 block Q(n).


### A2. Obtain $U$ and $C$ with RT
The first column is $e_0$. Subsequent columns select consecutive values of the scalar sequence. The forward cyclic rank is three although $M$ has four rows.

In [4]:
C_rt = MF.as_companion(n)
U_rt = Matrix(MF.companion_coboundary_matrix(n)[:, :C_rt.rows])
assert C_rt.shape == (3,3) and U_rt.shape == (4,3)
assert equal(C_rt,C) and equal(U_rt,U)
assert sm(M)*sm(U).shift(n,1) == sm(U)*sm(C)
e0 = Matrix.eye(4)[:,0]
assert U[:,0] == e0
assert equal(U[:,1], M*e0)
assert sm(U[:,2]) == sm(M)*sm(M).shift(n,1)*sm(e0)
print("PASS: forward reduced companion, Krylov columns, and coboundary.")

CQ = Q.as_companion(n)
UQ = Matrix(Q.companion_coboundary_matrix(n)[:,:CQ.rows])
assert equal(CQ,matrix("dual_companion.json","C"))
assert equal(UQ,matrix("dual_companion.json","U"))
assert sm(Q)*sm(UQ).shift(n,1) == sm(UQ)*sm(CQ)
print("PASS: dual U and C regenerated and checked.")

PASS: forward reduced companion, Krylov columns, and coboundary.


PASS: dual U and C regenerated and checked.


### A3. Select $\theta F$ instead of $F$
Swapping $e_0,e_1$ before companionization selects the second jet coordinate. Return to the original coordinates by multiplying the resulting $U$ by the permutation matrix.

In [5]:
P = Matrix.eye(4)
P.col_swap(0,1)
M_theta = M.coboundary(P,n)
C_theta = M_theta.as_companion(n)
V_theta = Matrix(M_theta.companion_coboundary_matrix(n)[:,:C_theta.rows])
U_theta = P*V_theta
assert U_theta[:,0] == Matrix.eye(4)[:,1]
assert sm(M)*sm(U_theta).shift(n,1) == sm(U_theta)*sm(C_theta)
print("PASS: theta-F companion; order =", C_theta.rows)

PASS: theta-F companion; order = 4


### A4. The connection to $Q^{-T}$
We verify $M(n)K(n+1)Q(n)^T=c(n)K(n)$ without forming an inverse. Together with invertibility of $Q$, this is the desired identity.

In [6]:
E = matrix("dual_matrices_zeta3_4.json","E")
H = matrix("dual_matrices_zeta3_4.json","H")
K = (sm(E)*sm(H)).factor()
dual = data("dual_matrices_zeta3_4.json")
c = expression(dual["A"])/expression(dual["B"])
assert equal(M,matrix("dual_matrices_zeta3_4.json","F"))
assert sm(M)*sm(K).shift(n,1)*sm(Q.T) == sm(c*K)
Delta = sp.prod(j*n+1 for j in range(8,13))
u_tilde = Matrix([1,10*n+1,(10*n+1)**2])
assert sm(K)*sm(u_tilde) == sm(Delta*e0)
K0 = K.subs({n:0})
assert K0 == Matrix([[0,0,1],[0,0,0],[-1,2,-1],[-2,2,0]])
print("PASS: dual connection and endpoint relation.")
display(K0)

PASS: dual connection and endpoint relation.


Matrix([
[ 0, 0,  1],
[ 0, 0,  0],
[-1, 2, -1],
[-2, 2,  0]])

### A5. No poles or rank loss at nonnegative integer indices
The sufficient polynomial test checks $p(0)\ne0$ and that $p(n+1)$ has coefficients of one sign. This proves nonvanishing at all integers $n\ge0$, not just a tested range.

In [7]:
# A shifted sign certificate needs a NONZERO constant term.
assert not regular_nonnegative(n-1)
assert not regular_nonnegative(n)
assert regular_nonnegative(n+1)
for A in (M,Q,U,C,UQ,CQ,K):
    for entry in A:
        den = sp.fraction(entry)[1]
        assert regular_nonnegative(den)
for A in (Q,C,CQ,UQ):
    num, den = sp.fraction(determinant(A))
    assert regular_nonnegative(num) and regular_nonnegative(den)
print("PASS: all listed matrices regular; Q, C, CQ, UQ invertible for n >= 0.")

PASS: all listed matrices regular; Q, C, CQ, UQ invertible for n >= 0.


**Verified here:** exact trajectories, both companions, coordinate change, dual intertwining, and regularity. Identifying a recessive analytic solution is a separate step; it is not implied by these algebraic identities alone.

## B. Sum presentations, integer normalization and the determinant formulas

**Paper:** Lemma 6.9 (`prop:z34-q-iv`), Lemma 6.10 (`prop:z34-p-iv`), Lemma 6.11 (`prop:z34-int`), Lemma 6.12 (`lem: dual sum`), and Lemma 6.13 (`prop:z34-dual`).

The normalization in the current manuscript is used exactly:
\[
q_n=12F_n,\qquad
\mathbf q_n=\frac{C_nq_n}{12}=\sum_{k=0}^{14n}B_{n,k},\qquad
\mathbf p_{s,n}=\frac{C_np_{s,n}}{12}.
\]
At $n=0$ this gives $q_0=12$, $\mathbf q_0=1$, and both numerator readouts zero. The small-index comparisons below are independent regression checks. They are supplemented by symbolic recurrence certificates in Notebook 2.


In [8]:
import math
import sympy as sp
from fractions import Fraction
from ramanujantools import Matrix
from common import n, matrix, data, expression, sm, determinant, scalar_equal, step, factorial_ratio_step
from readouts import B, Bprime, H, forward_values, dual_rows, normalization, Lam, lcm_to
M = matrix("matrices_for_zeta3_4.json","M")
G = matrix("dual_matrices_zeta3_4.json","G")
Q = Matrix(G[:3,:3])
R = Matrix([[12,0,0,0],[0,0,-6,3],[0,0,0,-2]])
S0 = Matrix([[3,0,0],[-2,-2,0],[1,1,1]])
assert Matrix(dual_rows(0)) == S0
assert forward_values(0) == [1,0,0]
print("PASS: initial rows and normalization at n=0.")
display(S0)

PASS: initial rows and normalization at n=0.


Matrix([
[ 3,  0, 0],
[-2, -2, 0],
[ 1,  1, 1]])

### B1. The explicit forward summand — Lemma 6.9 (`prop:z34-q-iv`)

Cancelling factorials in $C_nt_{n,k}$ gives
\[
B_{n,k}=\binom{k+12n}{10n}\binom{k+13n}{10n}\binom{k+14n}{10n}
         \binom{15n}{n+k}\binom{15n}{k}.
\]
Every factor is a positive integer on $0\le k\le14n$. The first value is $B_{n,0}=C_n$. Equality with $C_nt_{n,k}$ also follows by the identical consecutive-term ratio checked symbolically next. A terminating sum is evaluated explicitly; no numerical hypergeometric routine is used.

The numerator formula uses $-n\le k\le15n$. Although $B_{n,k}=0$ in its two tails, the derivative $B'_{n,k}$ need not vanish there. `Bprime` differentiates the Gamma interpolation before evaluating at integer $k$ and includes these tail terms.


In [9]:
k = sp.Symbol("k")
binomial_ratio = (sp.prod(k+j*n+1 for j in (12,13,14)) * (14*n-k)*(15*n-k)
                  / (sp.prod(k+j*n+1 for j in (2,3,4))*(n+k+1)*(k+1)))
hypergeometric_ratio = ((-14*n+k)*(-15*n+k)*sp.prod(1+j*n+k for j in (12,13,14))
                        / ((k+1)*sp.prod(1+j*n+k for j in (1,2,3,4))))
assert sp.cancel(binomial_ratio-hypergeometric_ratio) == 0
print("PASS: B(n,k+1)/B(n,k) = t(n,k+1)/t(n,k), identically in n,k.")


PASS: B(n,k+1)/B(n,k) = t(n,k+1)/t(n,k), identically in n,k.


### B2. Independent finite checks — Lemma 6.10 (`prop:z34-p-iv`) and Lemma 6.12 (`lem: dual sum`)
These exact comparisons are regression checks. Notebook 2 supplies all-$n$ telescoping certificates; a finite list of agreeing values alone does not prove the formulas.

In [10]:
WF, WQ = Matrix.eye(4), Matrix.eye(3)
for N in range(6):
    forward = Matrix([forward_values(N)])
    readout = normalization(N) * (R*WF)[:,0].T / 12
    assert forward == readout
    assert Matrix(dual_rows(N)) == S0*WQ
    D = 6*lcm_to(19*N)**4
    assert all((D*value).denominator == 1 for value in forward_values(N))
    if N < 5:
        WF = WF*M.subs({n:N})
        WQ = WQ*Q.subs({n:N})
print("PASS: all forward readouts, all 9 dual coordinates, and integer normalization, N=0,...,5.")

PASS: all forward readouts, all 9 dual coordinates, and integer normalization, N=0,...,5.


### The dual coefficients — Lemma 6.12 (`lem: dual sum`)

The exact kernel used by `laurent` and `dual_rows` is the paper's
\[
R_n(s)=(-1)^n\frac{(1+2n-s)_{14n}(1+n-s)_{16n}}
{(s)_{11n+1}(s+n)_{11n+1}(s+2n)_{8n+2}}.
\]
Set $J_n=\max(12n,10n+1)$. The Laurent coefficients of $(-s)^kR_n(s)$ at $s=-j$ are $c^{(k)}_{n,j,i}$, with $i=1,2,3$ and $k=0,1,2$. `dual_rows(n)` evaluates exactly
\[
\mathbf a_{n,k}=3\sum_jc^{(k)}_{n,j,3},\quad
\mathbf b_{n,k}=2\sum_jc^{(k)}_{n,j,2},\quad
\mathbf d_{n,k}=-\sum_{j,i}i\,c^{(k)}_{n,j,i}H^{(i+1)}_{2n+j}.
\]
At $n=0$, $R_0(s)=1/(s^3(s+1))$. The simple pole at $s=-1$ is included; omitting it would corrupt the initial constant row. The first comparison above checks all nine coordinates, including this exceptional initial case. Notebook 2 proves the row recurrences using three separate telescoping certificates.


### B3. Integer normalization for every $n$ — Lemma 6.11 (`prop:z34-int`)

Write $d_m=\operatorname{lcm}(1,\ldots,m)$ and $d_0=1$. We prove
\[
6d_{19n}^4(\mathbf q_n,\mathbf p_{3,n},\mathbf p_{4,n})\in\mathbb Z^3
\qquad(n\ge0).
\]

**Derivatives of binomials with variable upper argument.** The polynomial identity
\[
\frac{d}{dx}\binom{x}{m}=\sum_{j=1}^m\frac{(-1)^{j-1}}{j}\binom{x}{m-j}
\]
follows by differentiating $(1+t)^x$ and taking the coefficient of $t^m$. At integer $x$, every binomial on the right is integral. Thus $d_m$ clears the derivative's denominator. In our three variable-upper factors, $m=10n$.

**Fixed upper argument, including the tails.** For $N\ge0$ and $0\le x\le N$, Gamma interpolation gives
\[
\frac{d}{dx}\binom Nx=\binom Nx(H_{N-x}-H_x),
\]
whose denominator divides $d_N$. At an integer $x>N$, the derivative's magnitude is
\[
\frac{N!(x-N-1)!}{x!}=\frac1{x\binom{x-1}{N}}.
\]
The denominator divides $d_x$: in Legendre's formula each term
\[
\left\lfloor\frac{x}{p^r}\right\rfloor-
\left\lfloor\frac N{p^r}\right\rfloor-
\left\lfloor\frac{x-N-1}{p^r}\right\rfloor
\]
is zero or one and vanishes for $p^r>x$. The valuation is therefore at most $\lfloor\log_p x\rfloor=v_p(d_x)$. For $x<0$, use $\binom Nx=\binom N{N-x}$; differentiation changes the sign but not the denominator. This covers $N=0$ as well.

For $-n\le k\le15n$, the fixed-upper factors are $\binom{15n}{n+k}$ and $\binom{15n}{k}$. In-range derivatives require $d_{15n}$; reflected tail arguments are at most $16n$. The product rule yields exactly the paper's stronger intermediate statement
\[
B_{n,k}\in\mathbb Z,\qquad d_{16n}B'_{n,k}\in\mathbb Z.
\]
The undifferentiated factors remain integer valued even when zero. The derivative is taken before integer evaluation, so the simple-zero tails are retained.

**Harmonic denominators.** The index $4n+k$ lies in $[3n,19n]$, and
\[
d_{19n}^{\,r}H_m^{(r)}\in\mathbb Z\qquad(0\le m\le19n).
\]
Thus the two terms in $\mathbf p_{s,n}$ have denominators dividing $d_{19n}^{\,s}$ and $(s-1)d_{19n}^{\,s}$. For $s=3,4$, both are cleared by $6d_{19n}^4$. Also $\mathbf q_n$ is already an integer. At $n=0$, the triple is $(1,0,0)$.

This is the all-index proof. The next cell checks illustrative exact derivative identities and denominator examples; those finite checks serve as regression diagnostics.


In [11]:

import sympy as sp
from math import comb, lcm
from readouts import Bprime, forward_values

x = sp.Symbol('x')
def polynomial_binomial(x, m):
    return sp.prod(x-i for i in range(m)) / sp.factorial(m)

for m in range(1, 9):
    left = sp.diff(polynomial_binomial(x, m), x)
    right = sum(sp.Rational((-1)**(j-1),j)*polynomial_binomial(x,m-j)
                for j in range(1,m+1))
    assert sp.expand(left-right) == 0

for k in range(1, 61):
    dk = lcm(*range(1,k+1))
    for N in range(k):
        assert dk % (k*comb(k-1,N)) == 0

for N in range(5):
    d = lcm(*range(1,19*N+1)) if N else 1
    d16 = lcm(*range(1,16*N+1)) if N else 1
    for k in range(-N,15*N+1):
        assert (d16*Bprime(N,k)).denominator == 1
    assert all((6*d**4*v).denominator == 1 for v in forward_values(N))
print('Derivative identities, tail denominator divisibility, and sum checks: PASS')


Derivative identities, tail denominator divisibility, and sum checks: PASS


### B4. Recover the factorial determinant factor — Lemma 6.13 (`prop:z34-dual`)
For $S_a(n)=\prod_{j=1}^{a}(an+j)$, we verify the one-step factorization of $\det Q(n)$. Its product formula follows by telescoping and the value at $N=0$.

In [12]:
Delta = sp.prod(j*n+1 for j in range(8,13))
det_exp = {8:-1,9:-2,10:-3,11:-3,12:-3,13:-2,15:2,16:2,17:2,18:2,19:1}
detQ = determinant(Q)
expected_det = Delta/Delta.subs(n,n+1)*factorial_ratio_step(det_exp)
assert scalar_equal(detQ,expected_det)
dual = data("dual_matrices_zeta3_4.json")
kappa = expression(dual["A"])/expression(dual["B"])
kappa_exp = {15:1,1:1,2:1,3:1,4:1,13:-1,12:-1}
assert scalar_equal(kappa,factorial_ratio_step(kappa_exp))
print("PASS: determinant and kappa factorial-product identities.")

PASS: determinant and kappa factorial-product identities.


### B5. The factor $\Lambda_N$: $\Delta(N)$ cancels
$\Pi_N=(\prod_{m<N}c(m))/(\Delta(N)\det W_Q(N))$ and $\Lambda_N=C_N\Pi_N/6$. We add the factorial exponents, rather than assume the final expression.

In [13]:
C_exp = {12:1,13:1,15:1,10:-3,2:-1,3:-1,4:-1,1:-1}
lambda_exp = {a:C_exp.get(a,0)+kappa_exp.get(a,0)-det_exp.get(a,0)
              for a in set(C_exp)|set(kappa_exp)|set(det_exp)}
lambda_exp = {a:e for a,e in sorted(lambda_exp.items()) if e}
assert lambda_exp == {8:1,9:2,11:3,12:3,13:2,16:-2,17:-2,18:-2,19:-1}
assert sum(a*e for a,e in lambda_exp.items()) == 0
for N in range(6):
    derived = sp.Rational(1,6)*sp.prod(sp.factorial(a*N)**e for a,e in lambda_exp.items())
    assert derived == sp.Rational(Lam(N))
print("PASS: derived Lambda factorial exponents:",lambda_exp)
print("No Delta factor remains.")

PASS: derived Lambda factorial exponents: {8: 1, 9: 2, 11: 3, 12: 3, 13: 2, 16: -2, 17: -2, 18: -2, 19: -1}
No Delta factor remains.


### B6. Prove the algebraic hypotheses for the minor identities
For row vectors, $(yW)\times(zW)=\det(W)(y\times z)W^{-T}$. Combining this elementary identity with the intertwining proved in Part A proves the three minor formulas for every $N$. The initial cross products and the finite checks below fix all signs and factors.

In [14]:
K = (sm(matrix("dual_matrices_zeta3_4.json","E"))*sm(matrix("dual_matrices_zeta3_4.json","H"))).factor()
a,b,d = [S0[i,:] for i in range(3)]
assert R[0,:]*K.subs({n:0})/2 == b.cross(a)
assert -R[1,:]*K.subs({n:0})/2 == d.cross(a)
assert R[2,:]*K.subs({n:0})/2 == d.cross(b)
for N in range(6):
    a,b,d = [Matrix([r]) for r in dual_rows(N)]
    u = Matrix([[1,10*N+1,(10*N+1)**2]])
    minors = [Lam(N)*Matrix.vstack(b,a,u).det(),
              -Lam(N)*Matrix.vstack(d,a,u).det(),
              Lam(N)*Matrix.vstack(d,b,u).det()]
    assert minors == forward_values(N)
print("PASS: initial cross products and all three minor readouts at N=0,...,5.")

PASS: initial cross products and all three minor readouts at N=0,...,5.


### B7. Limit readouts — Lemma 6.7 (`lem:z34-limits`) at the initial point
This computation identifies the proposed constants **if** the analytic state is recessive and the hypotheses of the manuscript's dual-limit lemma hold. Notebook 3 separates the spectral calculation from that analytic hypothesis.

In [15]:
Z3,Z4 = sp.symbols("Z3 Z4")
initial_state = Matrix([1-2*Z3+3*Z4,1-2*Z3,1])
transported = R*K.subs({n:0})*initial_state
assert transported == Matrix([12,12*Z3,12*Z4])
print("PASS: proposed limits are Z3 and Z4, with nonzero denominator 12.")

PASS: proposed limits are Z3 and Z4, with nonzero denominator 12.


### What this notebook establishes

The trajectory reconstruction, dual pairing, companion identities, factorial multipliers and initial cross products have zero symbolic residuals. The integrality section gives a termwise argument valid for every $n$. The finite sum and minor comparisons are diagnostics. Their extension to all indices uses the regenerated recurrence certificates and boundary arguments in Notebook 2.

For Lemma 6.7 (`lem:z34-limits`), the exact initial calculation gives $(12,12\zeta(3),12\zeta(4))$. Applying the paper's Dual Convergence Theorem additionally requires the recessive-state and endpoint arguments developed in Notebook 3.
